<a href="https://colab.research.google.com/github/rojhatyigitdemir/BigData_Analytics/blob/main/DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%load_ext rpy2.ipython

In [ ]:
%%R
# 1. Install and load necessary packages
install.packages(c("DBI", "RSQLite", "arrow", "dplyr", "ggplot2"))
library(DBI)
library(RSQLite)
library(arrow)
library(dplyr)
library(ggplot2)

cat("1. Downloading CFPB complaints dataset...\n")
dataset_url <- "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
zip_file <- "complaints.csv.zip"

if (!file.exists(zip_file)) {
  download.file(dataset_url, destfile = zip_file)
}
if (!file.exists("complaints.csv")) {
  unzip(zip_file, exdir = ".")
}

db_file <- "lexalpha_risk.sqlite"
if (file.exists(db_file)) file.remove(db_file) # Clean old corrupted db if any

con <- dbConnect(RSQLite::SQLite(), db_file)

cat("2. Creating explicit SQLite table schema...\n")
create_table_sql <- "
CREATE TABLE complaints (
  date_received DATE,
  product TEXT,
  issue TEXT,
  company TEXT,
  state TEXT,
  tags TEXT,
  submitted_via TEXT,
  timely_response TEXT,
  is_untimely INTEGER
);
"
dbExecute(con, create_table_sql)

cat("3. Opening Arrow dataset for memory-safe streaming...\n")
ds <- open_dataset("complaints.csv", format = "csv")

target_cols <- c(
  "Date received", "Product", "Issue", "Company",
  "State", "Tags", "Submitted via", "Timely response?"
)

# Create a record batch reader to process data in chunks without crashing RAM
reader <- ds %>%
  select(all_of(target_cols)) %>%
  as_record_batch_reader()

cat("4. Streaming data into SQLite in safe batches (No RAM crash)...\n")
batch_count <- 0

repeat {
  batch <- reader$read_next_batch()
  if (is.null(batch)) break

  # Convert current batch to a small dataframe
  df_chunk <- as.data.frame(batch)

  # Clean and transform chunk
  names(df_chunk) <- c("date_received", "product", "issue", "company", "state", "tags", "submitted_via", "timely_response")

  df_chunk <- df_chunk %>%
    mutate(
      date_received = as.character(as.Date(date_received, format = "%m/%d/%Y")),
      tags = ifelse(is.na(tags) | tags == "", "Standard_Consumer", tags),
      is_untimely = ifelse(timely_response == "No", 1L, 0L)
    )

  # Append chunk directly to SQLite database
  dbWriteTable(con, "complaints", df_chunk, append = TRUE, row.names = FALSE)

  batch_count <- batch_count + 1
  if (batch_count %% 5 == 0) {
    cat(paste("Processed batch:", batch_count, "\n"))
  }

  # Aggressive memory cleanup per batch
  rm(df_chunk, batch)
  gc()
}

cat("SUCCESS: All batches streamed and written to SQLite!\n")

cat("5. Building SQL Indices for high-performance querying...\n")
dbExecute(con, "CREATE INDEX idx_company ON complaints (company);")
dbExecute(con, "CREATE INDEX idx_tags ON complaints (tags);")
dbExecute(con, "CREATE INDEX idx_date ON complaints (date_received);")

# Verify total row count
row_check <- dbGetQuery(con, "SELECT COUNT(*) as total_rows FROM complaints;")
print(row_check)

dbDisconnect(con)
cat("Database is fully ready and indexed!\n")

1. Downloading CFPB complaints dataset...
2. Creating explicit SQLite table schema...
3. Opening Arrow dataset for memory-safe streaming...
4. Streaming data into SQLite in safe batches (No RAM crash)...
Processed batch: 5 
Processed batch: 10 
Processed batch: 15 
Processed batch: 20 
Processed batch: 25 
Processed batch: 30 
Processed batch: 35 
Processed batch: 40 
Processed batch: 45 
Processed batch: 50 
Processed batch: 55 
Processed batch: 60 
Processed batch: 65 
Processed batch: 70 
Processed batch: 75 
Processed batch: 80 
Processed batch: 85 
Processed batch: 90 
Processed batch: 95 
Processed batch: 100 
Processed batch: 105 
Processed batch: 110 
Processed batch: 115 
Processed batch: 120 
Processed batch: 125 
Processed batch: 130 
Processed batch: 135 
Processed batch: 140 
Processed batch: 145 
Processed batch: 150 
Processed batch: 155 
Processed batch: 160 
Processed batch: 165 
Processed batch: 170 
Processed batch: 175 
Processed batch: 180 
Processed batch: 185 
Pr

Exception ignored from cffi callback <function _processevents at 0x7c77a82889a0>:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/rpy2/rinterface_lib/callbacks.py", line 283, in _processevents
    @ffi_proxy.callback(ffi_proxy._processevents_def,
KeyboardInterrupt: 


Processed batch: 3730 
Processed batch: 3735 
Processed batch: 3740 
Processed batch: 3745 
Processed batch: 3750 
Processed batch: 3755 
Processed batch: 3760 
Processed batch: 3765 
Processed batch: 3770 
Processed batch: 3775 
Processed batch: 3780 
Processed batch: 3785 
Processed batch: 3790 
Processed batch: 3795 
Processed batch: 3800 
Processed batch: 3805 
Processed batch: 3810 
Processed batch: 3815 
Processed batch: 3820 
Processed batch: 3825 
Processed batch: 3830 
Processed batch: 3835 
Processed batch: 3840 
Processed batch: 3845 
Processed batch: 3850 
Processed batch: 3855 
Processed batch: 3860 
Processed batch: 3865 
Processed batch: 3870 
Processed batch: 3875 
Processed batch: 3880 
Processed batch: 3885 
Processed batch: 3890 
Processed batch: 3895 
Processed batch: 3900 
Processed batch: 3905 
Processed batch: 3910 
Processed batch: 3915 
Processed batch: 3920 
Processed batch: 3925 
Processed batch: 3930 
Processed batch: 3935 
Processed batch: 3940 
Processed b